In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

# Start of Main Algorithm

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'kit(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    # Add WV_bin column based on WV_MeanPressure    
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base, df_data

## PREPROCESS DATA FOR ML

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

def preprocess_data(df, features, test_size=0.2, data_source_col="DataSource"):
    """
    Preprocess data for (un)supervised ML.

    - Filter WV_bin == 1
    - Split by data source:
        DataSource = 1 -> Monorail      (used for TRAIN and partly for TEST)
        DataSource = 0 -> Reference     (used ONLY for TEST)
    - X_train: only Monorail data (DataSource == 1)
    - X_test, y_test: 0.2 of Monorail + ALL Reference data
    - X_train_healthy: subset of X_train where label == 0

    Parameters
    ----------
    df : pd.DataFrame
        Full dataset.
    features : list of str
        Feature names to use.
    test_size : float, default 0.2
        Fraction of Monorail data to hold out for test.
    data_source_col : str, default "DataSource"
        Column indicating source (1 = Monorail, 0 = Reference).

    Returns
    -------
    X_train, X_test, y_train, y_test, X_train_healthy
    """

    df = df.copy()

    # ----------------------------
    # 1. Filter by WV_bin == 1
    # ----------------------------
    if "WV_bin" not in df.columns:
        raise ValueError("Column 'WV_bin' not found in dataframe.")
    df_filt = df[df["WV_bin"] == 1].copy()

    # ----------------------------
    # 2. Check features
    # ----------------------------
    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"The following features are not in the dataframe: {missing}")

    if data_source_col not in df_filt.columns:
        raise ValueError(f"Column '{data_source_col}' not found in dataframe.")

    if "label" not in df_filt.columns:
        raise ValueError("Column 'label' not found in dataframe.")

    # ----------------------------
    # 3. Split by data source
    # ----------------------------
    # Assume: 1 = Monorail, 0 = Reference
    df_mono = df_filt[df_filt[data_source_col] == 1].copy()
    df_ref  = df_filt[df_filt[data_source_col] == 0].copy()

    # Features & labels
    X_mono = df_mono[features]
    y_mono = df_mono["label"]

    X_ref  = df_ref[features]
    y_ref  = df_ref["label"]

    # ----------------------------
    # 4. Split Monorail into train / test
    # ----------------------------
    X_train_mono, X_test_mono, y_train_mono, y_test_mono = train_test_split(
        X_mono,
        y_mono,
        test_size=test_size,      # e.g. 0.2
        stratify=y_mono,
        random_state=42
    )

    # TRAIN = only Monorail
    X_train = X_train_mono
    y_train = y_train_mono

    # TEST = 0.2 Monorail + all Reference
    X_test = pd.concat([X_test_mono, X_ref], axis=0)
    y_test = pd.concat([y_test_mono, y_ref], axis=0)

    # Optional: shuffle test set so sources are mixed
    X_test, y_test = _shuffle_together(X_test, y_test, random_state=42)

    # ----------------------------
    # 5. Healthy-only training subset for anomaly models
    # ----------------------------
    X_train_healthy = X_train[y_train == 0]

    # ----------------------------
    # 6. Print summary
    # ----------------------------
    print(f"Total samples (all WV bins): {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")

    print("\n--- Monorail (DataSource=1, WV_bin==1) ---")
    print(f"Total Monorail: {len(df_mono)} "
          f"(Healthy: {(y_mono==0).sum()}, Leakage: {(y_mono==1).sum()})")

    print("\n--- Reference (DataSource=0, WV_bin==1) ---")
    print(f"Total Reference: {len(df_ref)} "
          f"(Healthy: {(y_ref==0).sum()}, Leakage: {(y_ref==1).sum()})")

    print("\n--- Train/Test split ---")
    print(f"Training samples (Monorail only): {len(X_train)} "
          f"(Healthy: {(y_train==0).sum()}, Leakage: {(y_train==1).sum()})")
    print(f"Training samples (healthy only, for anomaly): {len(X_train_healthy)}")
    print(f"Test samples (0.2 Monorail + all Reference): {len(X_test)} "
          f"(Healthy: {(y_test==0).sum()}, Leakage: {(y_test==1).sum()})")

    return X_train, X_test, y_train, y_test, X_train_healthy


def _shuffle_together(X, y, random_state=42):
    """Shuffle X and y in the same order."""
    X = X.copy()
    y = y.copy()
    rng = np.random.RandomState(random_state)
    idx = np.arange(len(X))
    rng.shuffle(idx)
    return X.iloc[idx], y.iloc[idx]


## Model Definition

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
# ============================================================================
# STEP 1: DEFINE UNSUPERVISED MODELS
# ============================================================================

from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor

def get_all_models(contamination=0.05):
    """
    Define ONLY unsupervised anomaly detection models.
    Returns dictionary of anomaly models.
    """

    models = {

        # ===========================================================
        # UNSUPERVISED MODELS (train on healthy-only data)
        # ===========================================================
        'anomaly': {
            'Isolation Forest': IsolationForest(
                contamination=contamination,
                n_estimators=200,
                random_state=42,
                n_jobs=-1
            ),

            'One-Class SVM': OneClassSVM(
                nu=contamination,
                kernel='rbf',
                gamma='scale'
            ),

            'Local Outlier Factor': LocalOutlierFactor(
                contamination=contamination,
                novelty=True,       # VERY IMPORTANT for test-set prediction
                n_neighbors=20
            ),
        }
    }

    return models

# ============================================================================
# STEP 2: TRAIN UNSUPERVISED MODELS
# ============================================================================

def train_all_models(X_train_scaled):
    """
    Train all unsupervised anomaly detection models.
    Input should be healthy-only samples.
    """

    models = get_all_models()
    trained_models = {}

    print("\n" + "="*60)
    print("TRAINING UNSUPERVISED MODELS")
    print("="*60)

    for name, model in models['anomaly'].items():
        print(f"  - Training {name}...")
        model.fit(X_train_scaled)

        trained_models[name] = {
            'model': model,
            'type': 'anomaly',
            'trained': True
        }

    print("\nAll anomaly detection models trained successfully!")
    return trained_models



### CROSS VALIDATION METHOD


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

def evaluate_anomaly_models(trained_models, X_test_scaled, y_test, positive_label=1):
    """
    Evaluate anomaly detection models using labeled test data.
    
    - Assumes model.predict -> +1 (normal), -1 (anomaly)
    - y_test: 0 = healthy, 1 = anomaly (combined leakage)
    """
    rows = []

    for name, info in trained_models.items():
        if info['type'] != 'anomaly':
            continue

        model = info['model']
        print(f"\nEvaluating {name}...")

        # -------------------------
        # 1) Hard predictions
        # -------------------------
        raw_pred = model.predict(X_test_scaled)   # +1 normal, -1 anomaly
        y_pred = (raw_pred == -1).astype(int)     # map to 1=anomaly

        cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        precision = precision_score(y_test, y_pred, pos_label=positive_label)
        recall    = recall_score(y_test, y_pred, pos_label=positive_label)
        f1        = f1_score(y_test, y_pred, pos_label=positive_label)

        # -------------------------
        # 2) Continuous scores
        #    (for ROC-AUC, PR-AUC)
        # -------------------------
        score = None

        if hasattr(model, "decision_function"):
            # For IF / OCSVM: higher score = more normal
            raw_score = model.decision_function(X_test_scaled)
            # We want higher value = more anomalous for ROC/PR, so flip sign
            score = -raw_score  

        elif hasattr(model, "score_samples"):
            raw_score = model.score_samples(X_test_scaled)
            score = -raw_score

        # If score is available, compute ROC-AUC and PR-AUC
        if score is not None:
            roc_auc = roc_auc_score(y_test, score)
            pr_auc  = average_precision_score(y_test, score)
        else:
            roc_auc = np.nan
            pr_auc  = np.nan

        rows.append({
            "Model": name,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc,
            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn
        })

        print(f"  Precision (anomaly): {precision:.3f}")
        print(f"  Recall    (anomaly): {recall:.3f}")
        print(f"  F1        (anomaly): {f1:.3f}")
        print(f"  ROC-AUC   (anomaly score): {roc_auc:.3f}")
        print(f"  PR-AUC    (anomaly score): {pr_auc:.3f}")

    results = pd.DataFrame(rows)
    return results


### Test Metrics Table

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_table(trained_models, cv_summary, X_test, y_test):
    """
    Evaluate tuned models on a TEST set using the thresholds
    previously selected via CV on the TRAIN set.

    Parameters
    ----------
    trained_models : dict
        Same structure you used before: {name: {'model': ..., 'type': ...}}.
        These models are already fitted on the TRAIN data.
    cv_summary : pd.DataFrame
        Output of cv_metrics_table on the TRAIN set, containing
        at least columns ['Model', 'BestThreshold'].
    X_test, y_test : array-like
        Held-out test data (no CV here).

    Returns
    -------
    pd.DataFrame
        Metrics on the test set for each model, using its CV-tuned threshold.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        if mtype not in ["supervised", "supervised_smote"]:
            # skip anomaly / unsupervised etc.
            continue

        # 1) Get the best threshold from TRAIN CV summary
        row = cv_summary.loc[cv_summary["Model"] == name]
        if row.empty:
            # model was not in cv_summary_1 (or got filtered out)
            continue
        best_threshold = row["BestThreshold"].values[0]

        # 2) Predict probabilities on the TEST set
        #    (for SMOTE pipelines, you should have refit the pipeline on TRAIN
        #     before building `trained_models`).
        y_proba = base_model.predict_proba(X_test)[:, 1]

        # 3) Apply threshold
        y_pred = (y_proba >= best_threshold).astype(int)

        # 4) Compute metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y_test, y_proba)

        rows.append({
            "Model": name,
            "Type": mtype,
            "BestThreshold": best_threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    return pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_from_final(final_models, final_thresholds, X_test, y_test):
    """
    Evaluate already-fitted final models on TEST set, using
    per-model thresholds that were tuned on the TRAIN set.

    Parameters
    ----------
    final_models : dict
        {model_name: fitted_estimator}
        (e.g. final_models_S1, final_models_S2, ...)
    final_thresholds : dict
        {model_name: best_threshold_from_train}
        (e.g. final_thresholds_S1, ...)
    X_test, y_test : array-like
        Held-out test set.

    Returns
    -------
    pd.DataFrame
        Test metrics for each model.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, estimator in final_models.items():
        if name not in final_thresholds:
            continue

        thr = final_thresholds[name]

        # probabilities on TEST
        proba = estimator.predict_proba(X_test)[:, 1]
        y_pred = (proba >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        roc  = roc_auc_score(y_test, proba)

        rows.append({
            "Model": name,
            "BestThreshold_train": thr,
            "Precision_Test": prec,
            "Recall_Test": rec,
            "F1_Test": f1,
            "ROC-AUC_Test": roc,
            "TP_Test": tp,
            "FP_Test": fp,
            "FN_Test": fn,
            "TN_Test": tn,
        })

    return pd.DataFrame(rows).sort_values(
        by="F1_Test", ascending=False
    ).reset_index(drop=True)


### Model Hyperparameter Tuning
Unsupervised Model

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)
from itertools import product

def tune_anomaly_model(model_class, param_grid, 
                       X_train_healthy_scaled, 
                       X_val_scaled, y_val,
                       model_name="Model"):
    """
    Simple grid search for anomaly detection models.

    model_class : class (e.g., IsolationForest, OneClassSVM)
    param_grid  : dict of param -> list of values
    X_train_healthy_scaled : np.array, healthy-only training data
    X_val_scaled, y_val    : validation data with labels (0 healthy, 1 faulty)
    """
    results = []

    # build all combinations of params
    keys = list(param_grid.keys())
    combos = list(product(*param_grid.values()))

    print(f"\nTUNING {model_name}")
    print(f"Total combinations: {len(combos)}")

    for combo in combos:
        params = dict(zip(keys, combo))
        print(f"  - Trying params: {params}")

        # instantiate and fit
        model = model_class(**params)
        model.fit(X_train_healthy_scaled)

        # predict on validation set (+1 normal, -1 anomaly)
        raw_pred = model.predict(X_val_scaled)
        y_pred = (raw_pred == -1).astype(int)  # 1 = anomaly, 0 = healthy

        # confusion matrix
        cm = confusion_matrix(y_val, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()

        # metrics for anomaly class (label=1)
        precision = precision_score(y_val, y_pred, pos_label=1)
        recall    = recall_score(y_val, y_pred, pos_label=1)
        f1        = f1_score(y_val, y_pred, pos_label=1)

        # anomaly scores for ROC/PR if possible
        score = None
        if hasattr(model, "decision_function"):
            raw_score = model.decision_function(X_val_scaled)
            score = -raw_score   # flip so higher = more anomalous
        elif hasattr(model, "score_samples"):
            raw_score = model.score_samples(X_val_scaled)
            score = -raw_score

        if score is not None:
            roc_auc = roc_auc_score(y_val, score)
            pr_auc  = average_precision_score(y_val, score)
        else:
            roc_auc = np.nan
            pr_auc = np.nan

        row = {
            "Model": model_name,
            **params,
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
            "ROC-AUC": roc_auc,
            "PR-AUC": pr_auc,
            "TP": tp, "FP": fp, "FN": fn, "TN": tn
        }
        results.append(row)

    results_df = pd.DataFrame(results)

    # pick best by F1
    best_idx = results_df["F1"].idxmax()
    best_params = results_df.loc[best_idx, keys].to_dict()
    best_f1 = results_df.loc[best_idx, "F1"]

    print(f"\nBest {model_name} params by F1 on validation:")
    print(best_params)
    print(f"Best F1 (anomaly): {best_f1:.3f}")

    return results_df, best_params


# Load Data

In [ ]:
# ============================================================================
# STEP 0: DATA LOADING 
# ============================================================================

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_kit01.csv',
    'TestBrakefinal_data_kit06.csv',
    'TestBrakefinal_data_kit27.csv'
]

[df, df_monorail] = load_data(model_path, monorail_paths)

print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

print(df_monorail.shape)

In [ ]:
df.head()

In [ ]:
df_monorail.head()

## Data Exploration Continues

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df['y_jitter'] = rng.normal(0, 0.02, size=len(df))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}

# df_filtered = df[(df['EmergencyBrake_action'] == 1) & (df['WV_bin'] == 1)].copy()
df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['WV_bin'] == 1)].copy()
# df_filtered = df.copy()
# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Total_power_efficiency'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Efficiency distribution by Kit Source")
axes[0].set_xlabel("Total Power Efficiency")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")
axes[0].set_xlim(0, 8)
# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Total_power_efficiency'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Efficiency distribution by Label")
axes[1].set_xlabel("Total Power Efficiency")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(0, 8)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df['y_jitter'] = rng.normal(0, 0.02, size=len(df))
# df_filtered = df[(df['Max_pressure_pipe'] < 0.8) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['EmergencyBrake_action'] == 0) & (df['WV_bin'] == 1)].copy()
df_filtered = df[
    (df['DataSource'] == 0) & 
    (df['WV_bin'].isin([1]))
].copy()
# df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['WV_bin'] == 1)].copy()
# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot Total_power_efficiency against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'Total_power_efficiency', 'label', and selected features.
    features : list of str
        List of feature column names to plot against Total_power_efficiency.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (Total_power_efficiency).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'Total_power_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'Total_power_efficiency'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )

        plt.xlabel("Total_power_efficiency")
        plt.ylabel(feature)
        plt.title(f"Total_power_efficiency vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:
df_filtered = df[df['WV_bin'] == 1]

selected_features = ["Total_power_delay", "WV_MeanPressure","Total_energy_efficiency","Total_power_efficiency"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, selected_features, x_limits=(0, 40))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_filt = df[df["WV_bin"] == 1].copy()
df_filt['y_jitter'] = rng.normal(0, 0.02, size=len(df_filt))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df_filt['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filt['Source'].astype('category').cat.codes
label_codes  = df_filt['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filt['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filt['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")

plt.tight_layout()
plt.show()


In [ ]:
# === Unified Feature Selection Pipeline: RF, MI, ANOVA
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.feature_selection import mutual_info_classif, f_classif, RFE, RFECV
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.feature_selection import VarianceThreshold
imp = SimpleImputer(strategy="median")
Features = df.copy().drop(columns=['label','Source','DataSource','WV_bin'])
Features = pd.DataFrame(imp.fit_transform(Features), columns=Features.columns, index=Features.index)
Target   = df['label']
Xsel = Features.copy()

# Keep only numeric columns (if any non-numeric slipped in)
num_cols = [c for c in Xsel.columns if np.issubdtype(Xsel[c].dtype, np.number)]
Xsel = Xsel[num_cols].copy()

# Example: drop features with variance below 1e-2 AFTER scaling (optional):
vt = VarianceThreshold(threshold=1e-2)
X_var = vt.fit_transform(Xsel)
kept_mask = vt.get_support()
kept_features = Xsel.columns[kept_mask]
Xsel = Xsel[kept_features]


# Some selectors need scaling
sc_std = StandardScaler()
sc_rob = RobustScaler()
X_std = pd.DataFrame(sc_std.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)
X_rob = pd.DataFrame(sc_rob.fit_transform(Xsel), columns=Xsel.columns, index=Xsel.index)

# if y is string, change to 0/1
y_enc = pd.Series(Target).astype("category")
if y_enc.dtype.name == "category":
    y_enc = y_enc.cat.codes  # e.g., Leakage=1, Normal=0

# For stability on tiny datasets
cv = StratifiedKFold(n_splits=min(5, max(2, np.bincount(y_enc).min())), shuffle=True, random_state=42)

# Helper to convert scores to ranks (lower rank = better)
def to_rank(series, higher_is_better=True):
    s = series.copy()
    if not higher_is_better:
        s = -s
    # rank 1 = best
    return s.rank(ascending=False, method="average")

# ---------- 1) RandomForest importance ----------
# Measures a feature's utility in improving the model's prediction accuracy (e.g., mean decrease in impurity).
# Captures feature interactions naturally; highly effective.
# REDUCED trees for small dataset stability
rf = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_leaf=3, 
                            random_state=42, class_weight="balanced")
rf.fit(Xsel, y_enc)
rf_imp = pd.Series(rf.feature_importances_, index=Xsel.columns, name="RF_Importance")
rf_rank = to_rank(rf_imp, higher_is_better=True).rename("RF_Rank")

# ---------- 2) Mutual Information ----------
# Measures statistical dependency (information gain) between a feature and the target.
# Captures non-linear relationships. Evaluates each feature independently; ignores feature interactions.
mi = mutual_info_classif(Xsel, y_enc, random_state=42, discrete_features=False, n_neighbors=3)
mi_score = pd.Series(mi, index=Xsel.columns, name="MI_Score")
mi_rank = to_rank(mi_score, higher_is_better=True).rename("MI_Rank")

# ---------- 3) ANOVA F-test ----------
# (works best if roughly Gaussian/scaled; we used imputed data)
# Measures linear correlation between a feature and the target by comparing variance between groups to variance within groups.
# Assumes linear relationship and Gaussian distribution; ignores feature interactions.
F_vals, p_vals = f_classif(Xsel, y_enc)
f_score = pd.Series(F_vals, index=Xsel.columns, name="ANOVA_F")
f_rank = to_rank(f_score, higher_is_better=True).rename("ANOVA_Rank")

# ---------- Combine all rankings with OPTIMIZED WEIGHTS for 60 samples ----------
rank_table = pd.concat([rf_rank, mi_rank, f_rank,
                        rf_imp, mi_score, f_score], axis=1)

# WEIGHTED OverallRank for small datasets (60 samples)
# MI: 0.50 (highest weight - most reliable for small data)
# ANOVA: 0.30 (second - stable if linear relationships exist)
# RF: 0.20 (lowest - prone to overfitting with 60 samples)
weights = {
    "MI_Rank": 0.70,
    "ANOVA_Rank": 1,
    "RF_Rank": 0.20
}

rank_table["OverallRank"] = (
    rank_table["MI_Rank"] * weights["MI_Rank"] +
    rank_table["ANOVA_Rank"] * weights["ANOVA_Rank"] +
    rank_table["RF_Rank"] * weights["RF_Rank"]
)

# Sort and display top-N
N = 10
rank_table_sorted = rank_table.sort_values("OverallRank").head(N)
print("=== Top features by Weighted OverallRank (lower = better) ===")
print(f"Weights: MI={weights['MI_Rank']}, ANOVA={weights['ANOVA_Rank']}, RF={weights['RF_Rank']}")
display(rank_table_sorted)

plt.figure(figsize=(8, max(4, 0.35*N)))
rank_table_sorted.sort_values("OverallRank")["OverallRank"].plot(kind="barh")
plt.gca().invert_yaxis()
plt.title(f"Top {N} Features by (Weight more on Anova)")
plt.xlabel("Rank (lower is better)")
plt.tight_layout()
plt.show()

topN_features = rank_table_sorted.index.tolist()
print("\nTopN feature list:", topN_features)

Features_reduced = Features[topN_features]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.heatmap(Features_reduced.corr(), annot=True, cmap='coolwarm')
plt.title("Feature Correlation Heatmap")
plt.show()

## Algorithm 1 - Use Features that is Important

In [ ]:
# ============================================================================
# STEP 1: DATA PREPROCESSING TRAIN TEST SPLIT AND FEATURE SELECTION 
# ============================================================================
selected_features = ['Std_delay_exp','Total_power_delay','Total_power_efficiency'] 

[X_train_1, X_test_1, y_train, y_test, X_train_healthy_1] = preprocess_data(df, selected_features, test_size=0.2, data_source_col="DataSource")
X_train_1.head()

In [ ]:
# ============================================================================
# STEP 2: IMPUTE + FEATURE SCALING
# ============================================================================

from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

[X_train_scaled_1, X_test_scaled_1, X_train_healthy_scaled_1, scaler, imputer] = scale_features(X_train_1, X_test_1, X_train_healthy_1)

### Train the Model

In [ ]:
trained_models = train_all_models(X_train_healthy_scaled_1)

### Simple model testing

In [ ]:
iso = trained_models["Isolation Forest"]["model"]
svm = trained_models["One-Class SVM"]["model"]

### Simple Model Training

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# y_test: 0 = healthy, 1 = leakage
# ISO
raw_pred_iso = iso.predict(X_test_scaled_1)   # +1 normal, -1 anomaly
y_pred_iso = (raw_pred_iso == -1).astype(int) # 1 = anomaly, 0 = healthy

# SVM
raw_pred_svm = svm.predict(X_test_scaled_1) 
y_pred_svm = (raw_pred_svm == -1).astype(int)

# Confusion matrices
cm_iso = confusion_matrix(y_test, y_pred_iso)
cm_svm = confusion_matrix(y_test, y_pred_svm)

print("Isolation Forest CM:\n", cm_iso)
print("One-Class SVM CM:\n", cm_svm)

# Basic metrics (treat 1 = anomaly as positive class)
for name, y_pred in [("Isolation Forest", y_pred_iso),
                     ("One-Class SVM", y_pred_svm)]:
    prec = precision_score(y_test, y_pred, pos_label=1)
    rec  = recall_score(y_test, y_pred, pos_label=1)
    f1   = f1_score(y_test, y_pred, pos_label=1)
    print(f"\n{name}:")
    print(f"  Precision (anomaly): {prec:.3f}")
    print(f"  Recall    (anomaly): {rec:.3f}")
    print(f"  F1        (anomaly): {f1:.3f}")


In [ ]:
results_anomaly = evaluate_anomaly_models(trained_models, X_test_scaled_1, y_test)
results_anomaly

Try the UNTUNNED MODEL on Test Data